# Is home-field advantage different by team?

For each team, home-field advantage ≈ (average margin at home − average margin on the road) / 2.
The team's own strength cancels out, leaving the home boost.

A single season gives only ~8 home games, so each team's estimate has about ±3.4 points of noise.
The question isn't which teams rank highest (there's always a ranking) but whether a team's
home field in one period **predicts** it in the next.

In [3]:
import pandas as pd
from nflmodel.data.games import load_games

g = load_games()
g = g[~g["neutral"]]

def team_hfa(games):
    home = games.groupby("home_team")["result"].mean()       # team's margin at home
    away = -games.groupby("away_team")["result"].mean()      # team's margin on the road
    return ((home - away) / 2).rename("hfa")

early = team_hfa(g[g["season"].between(2006, 2015)])
late = team_hfa(g[g["season"].between(2016, 2025)])
both = pd.concat([early.rename("2006-15"), late.rename("2016-25")], axis=1)

print(f"League HFA: early {early.mean():.2f}, late {late.mean():.2f}")
print(f"Correlation between periods: {both.corr().iloc[0, 1]:.2f}")
print(both.sort_values("2016-25", ascending=False).round(2).to_string())

a = team_hfa(g[g["season"].between(2018, 2021)])
b = team_hfa(g[g["season"].between(2022, 2025)])
print(f"Correlation 2018-21 vs 2022-25: {pd.concat([a, b], axis=1).corr().iloc[0, 1]:.2f}")

# and every pair of back-to-back seasons
seasons = range(2007, 2026)
corrs = [pd.concat([team_hfa(g[g["season"] == s - 1]), team_hfa(g[g["season"] == s])], axis=1)
         .corr().iloc[0, 1] for s in seasons]
print(f"Average season-to-season correlation: {sum(corrs) / len(corrs):.2f}")

League HFA: early 2.39, late 1.93
Correlation between periods: -0.19
           2006-15  2016-25
home_team                  
MIA           0.51     4.43
CHI           1.68     3.67
DEN           2.22     3.21
GB            3.50     3.12
NYJ           2.31     3.08
LV            1.14     3.03
CLE           2.05     3.00
MIN           2.69     2.69
PIT           3.28     2.67
PHI           1.37     2.59
BUF           3.11     2.59
HOU           2.97     2.56
TEN           1.46     2.39
JAX           3.13     2.23
CAR           1.33     2.15
IND           3.65     2.14
DET           3.58     1.86
DAL           1.61     1.80
ATL           2.38     1.67
SF            3.57     1.52
KC            0.81     1.51
NE            2.46     1.35
SEA           4.63     1.23
CIN           1.93     1.20
BAL           3.69     0.88
TB            1.04     0.78
NYG           1.22     0.62
LA            2.44     0.56
ARI           3.58     0.47
LAC           2.89     0.43
WAS           1.24     0.15
NO     

**Result:** team estimates don't persist at any time scale: −0.19 (10-year windows), 0.13 (4-year),
and −0.07 (season to season). Differences between teams are noise, so the model uses one
league-wide home-field term. League-wide home field did fall, from 2.39 to 1.93 points.